## Build Drivers Dimension

1. Read silver drivers table
2. Read gold ref_nationality_region table
3. Join the data from drivers with ref_nationality_region using nationality
4. Select the required columns
    - drivers.drivers_id
    - drivers.drivers_name
    - drivers.date_of_birth
    - drivers.nationality
    - ref_nationality_region.region
5. Write the transformed data to gold dim_drivers table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/04.gold-helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

### Step 1 -  Read silver drivers table


In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers").filter((col("batch_id") == v_batch_id))


### Step 2 - Read gold ref_nationality_region table

In [0]:
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")


### Step 3 - Join the data from drivers with ref_nationality_region using nationality


In [0]:
dim_drivers_df = drivers_df.join(ref_nationality_region_df, drivers_df.nationality == ref_nationality_region_df.nationality, "left")

### Step 4. Select the required columns
-     drivers.drivers_id
-     drivers.drivers_name
-     drivers.date_of_birth
-     drivers.nationality
-     ref_nationality_region.region

In [0]:
dim_drivers_df = dim_drivers_df.select(
    drivers_df.driver_id,
    drivers_df.driver_name,
    drivers_df.date_of_birth,
    drivers_df.nationality,
    ref_nationality_region_df.region.alias("nationality_region")
)

### Step 5 - Write the transformed data to gold dim_drivers table

In [0]:
# (
#     dim_drivers_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(target_table)
# )

In [0]:
write_to_gold(
    dim_drivers_df,
    target_table,
    "t.driver_id = s.driver_id",
    [
        "driver_name",
        "date_of_birth",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))